In [ ]:
import os
import json
import requests

PROJECT_NAME = "adam_and_eve"

# TEXT_SOURCE: local file path or URL
# e.g. "source_texts/adam_and_eve/book.txt"
# e.g. "https://www.gutenberg.org/files/84/84-0.txt"
TEXT_SOURCE = ""

OUTPUT_FILE = f"finetuning_data/{PROJECT_NAME}/source_sections.jsonl"

# Split text on double-newlines; skip lines shorter than this
MIN_PARA_LENGTH = 40

# Paragraphs visible at a time in the browser
WINDOW_SIZE = 20

In [ ]:
def load_text(source: str) -> str:
    if source.startswith("http"):
        r = requests.get(source, timeout=30)
        r.raise_for_status()
        return r.text
    with open(source, encoding="utf-8", errors="replace") as f:
        return f.read()

if not TEXT_SOURCE:
    raise ValueError("Set TEXT_SOURCE to a file path or URL")

raw_text = load_text(TEXT_SOURCE)
paragraphs = [
    p.strip()
    for p in raw_text.split("\n\n")
    if len(p.strip()) >= MIN_PARA_LENGTH
]
print(f"Loaded {len(paragraphs)} paragraphs from {TEXT_SOURCE}")
print(f"First paragraph: {paragraphs[0][:120]}...")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


class TextBrowserUI:
    def __init__(self, paragraphs, output_file, source_name, window_size=20):
        self.paragraphs = paragraphs
        self.n = len(paragraphs)
        self.output_file = output_file
        self.source_name = source_name
        self.WINDOW = window_size
        self.saved = []
        self._window_start = 0
        self._last_search = ""
        self._last_search_idx = -1

        # --- Para display ---
        self.para_display = widgets.HTML(
            layout=widgets.Layout(
                height="480px", overflow_y="scroll",
                border="1px solid #dee2e6", padding="8px", width="100%",
            )
        )
        self.page_label = widgets.Label()
        self.prev_btn = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="90px"))
        self.next_btn = widgets.Button(description="Next ▶", layout=widgets.Layout(width="90px"))

        # --- Search ---
        self.search_input = widgets.Text(
            placeholder="Keyword search (Enter or click Find)",
            layout=widgets.Layout(width="380px"),
        )
        self.search_btn = widgets.Button(
            description="Find", button_style="info", layout=widgets.Layout(width="70px")
        )
        self.search_status = widgets.Label()

        # --- Selection sliders ---
        self.start_slider = widgets.IntSlider(
            value=0, min=0, max=self.n - 1,
            description="Start:",
            continuous_update=False,
            style={"description_width": "50px"},
            layout=widgets.Layout(width="680px"),
        )
        self.end_slider = widgets.IntSlider(
            value=min(4, self.n - 1), min=0, max=self.n - 1,
            description="End:",
            continuous_update=False,
            style={"description_width": "50px"},
            layout=widgets.Layout(width="680px"),
        )

        # --- Preview ---
        self.preview = widgets.HTML(
            layout=widgets.Layout(
                height="280px", overflow_y="scroll",
                border="1px solid #28a745", padding="10px", width="100%",
                background="#f6fff8",
            )
        )

        # --- Actions ---
        self.note_input = widgets.Text(
            placeholder="Why is this section good/bad? (optional note)",
            layout=widgets.Layout(width="100%"),
        )
        self.add_chosen_btn = widgets.Button(
            description="Add as Chosen ✓", button_style="success",
            layout=widgets.Layout(width="180px"),
        )
        self.add_rejected_btn = widgets.Button(
            description="Add as Rejected ✗", button_style="danger",
            layout=widgets.Layout(width="185px"),
        )
        self.save_btn = widgets.Button(
            description="Save to JSONL", button_style="warning",
            layout=widgets.Layout(width="140px"),
        )
        self.counter = widgets.HTML()
        self.msg = widgets.Output()

        # --- Wire up events ---
        self.prev_btn.on_click(lambda _: self._shift(-self.WINDOW))
        self.next_btn.on_click(lambda _: self._shift(+self.WINDOW))
        self.search_btn.on_click(self._search)
        self.search_input.on_submit(self._search)
        self.start_slider.observe(self._on_slider, names="value")
        self.end_slider.observe(self._on_slider, names="value")
        self.add_chosen_btn.on_click(lambda _: self._add("chosen"))
        self.add_rejected_btn.on_click(lambda _: self._add("rejected"))
        self.save_btn.on_click(self._save)

        self._refresh_display()
        self._refresh_preview()
        self._refresh_counter()

    # -----------------------------------------------------------------------
    # Rendering helpers
    # -----------------------------------------------------------------------

    def _refresh_display(self, search_term=None):
        ws = self._window_start
        we = min(ws + self.WINDOW, self.n)
        start, end = self.start_slider.value, self.end_slider.value

        html = '<div style="font-family: Georgia, serif;">'
        for i in range(ws, we):
            selected = start <= i <= end
            bg = "#fff9e6" if selected else "#ffffff"
            border = "#ffc107" if selected else "#e9ecef"
            text = self.paragraphs[i]
            if search_term and search_term.lower() in text.lower():
                idx = text.lower().find(search_term.lower())
                text = (
                    text[:idx]
                    + f'<mark style="background:#ffe066">{text[idx:idx+len(search_term)]}</mark>'
                    + text[idx + len(search_term):]
                )
            html += (
                f'<div style="background:{bg}; border-left:3px solid {border}; '
                f'padding:5px 8px; margin:2px 0; border-radius:2px;">'
                f'<span style="color:#adb5bd; font-size:11px; font-family:monospace; user-select:none;">'
                f'[{i:4d}] </span>'
                f'<span style="font-size:13px; line-height:1.5;">{text}</span>'
                f'</div>'
            )
        html += "</div>"
        self.para_display.value = html
        self.page_label.value = f"Paras {ws}–{we-1} of {self.n-1}"

    def _refresh_preview(self):
        start, end = self.start_slider.value, self.end_slider.value
        if start > end:
            self.preview.value = '<p style="color:red">Start must be ≤ End</p>'
            return
        text = "\n\n".join(self.paragraphs[start : end + 1])
        wc = len(text.split())
        body = text.replace("\n\n", "<br><br>").replace("\n", " ")
        self.preview.value = (
            f'<p style="color:#6c757d; font-size:11px; font-family:monospace; margin:0 0 6px;">'
            f'Paragraphs {start}–{end} &nbsp;·&nbsp; {end - start + 1} paragraphs &nbsp;·&nbsp; {wc} words</p>'
            f'<hr style="margin:4px 0;">'
            f'<div style="font-family:Georgia,serif; font-size:13px; line-height:1.6;">{body}</div>'
        )

    def _refresh_counter(self):
        nc = sum(1 for s in self.saved if s["type"] == "chosen")
        nr = sum(1 for s in self.saved if s["type"] == "rejected")
        self.counter.value = (
            f'<b style="color:green">✓ {nc} chosen</b>'
            f'&nbsp;&nbsp;'
            f'<b style="color:#dc3545">✗ {nr} rejected</b>'
        )

    # -----------------------------------------------------------------------
    # Event handlers
    # -----------------------------------------------------------------------

    def _shift(self, delta):
        self._window_start = max(0, min(self.n - self.WINDOW, self._window_start + delta))
        self._refresh_display()

    def _search(self, _):
        term = self.search_input.value.strip()
        if not term:
            return
        matches = [i for i, p in enumerate(self.paragraphs) if term.lower() in p.lower()]
        if not matches:
            self.search_status.value = f'No matches for "{term}"'
            return

        # Cycle through matches on repeated searches for the same term
        if term == self._last_search:
            after = [m for m in matches if m > self._last_search_idx]
            idx = after[0] if after else matches[0]
        else:
            idx = matches[0]
        self._last_search = term
        self._last_search_idx = idx

        match_num = matches.index(idx) + 1
        self.search_status.value = f"Match {match_num}/{len(matches)}"
        self._window_start = max(0, idx - 2)
        self.start_slider.value = idx
        self.end_slider.value = min(idx + 3, self.n - 1)
        self._refresh_display(search_term=term)

    def _on_slider(self, _):
        start = self.start_slider.value
        # Scroll display window to include start of selection
        if not (self._window_start <= start < self._window_start + self.WINDOW):
            self._window_start = max(0, start - 2)
        self._refresh_display()
        self._refresh_preview()

    def _add(self, section_type: str):
        start, end = self.start_slider.value, self.end_slider.value
        if start > end:
            with self.msg:
                clear_output()
                print("Error: Start must be ≤ End")
            return
        text = "\n\n".join(self.paragraphs[start : end + 1])
        self.saved.append({
            "type": section_type,
            "text": text,
            "source": self.source_name,
            "para_start": start,
            "para_end": end,
            "note": self.note_input.value,
        })
        self._refresh_counter()
        self.note_input.value = ""
        icon = "✓" if section_type == "chosen" else "✗"
        with self.msg:
            clear_output()
            print(f"{icon} {section_type.capitalize()} added (paras {start}–{end}, {len(text.split())} words)")

    def _save(self, _):
        if not self.saved:
            with self.msg:
                clear_output()
                print("Nothing to save yet.")
            return
        os.makedirs(os.path.dirname(self.output_file) or ".", exist_ok=True)
        with open(self.output_file, "w") as f:
            for entry in self.saved:
                f.write(json.dumps(entry) + "\n")
        nc = sum(1 for s in self.saved if s["type"] == "chosen")
        nr = sum(1 for s in self.saved if s["type"] == "rejected")
        with self.msg:
            clear_output()
            print(f"Saved {len(self.saved)} sections ({nc} chosen, {nr} rejected) → {self.output_file}")

    # -----------------------------------------------------------------------
    # Layout
    # -----------------------------------------------------------------------

    def show(self):
        display(
            widgets.VBox([
                widgets.HTML(
                    f'<h3 style="margin-bottom:4px">Text Browser</h3>'
                    f'<p style="color:#6c757d; margin-top:0">{self.source_name} — {self.n} paragraphs</p>'
                ),
                widgets.HBox([self.search_input, self.search_btn, self.search_status]),
                self.para_display,
                widgets.HBox([
                    self.prev_btn,
                    widgets.HTML("&nbsp;"),
                    self.page_label,
                    widgets.HTML("&nbsp;"),
                    self.next_btn,
                ]),
                widgets.HTML('<hr style="margin:10px 0"><h4 style="margin-bottom:4px">Select Range</h4>'),
                self.start_slider,
                self.end_slider,
                self.preview,
                widgets.HTML('<p style="font-size:12px; color:#6c757d; margin:6px 0">Optional: note why this section is interesting before adding</p>'),
                self.note_input,
                widgets.HBox([self.add_chosen_btn, widgets.HTML("&nbsp;"), self.add_rejected_btn]),
                widgets.HTML('<hr style="margin:10px 0">'),
                widgets.HBox([self.counter, widgets.HTML("&nbsp;&nbsp;&nbsp;"), self.save_btn]),
                self.msg,
            ])
        )


browser = TextBrowserUI(paragraphs, OUTPUT_FILE, TEXT_SOURCE, window_size=WINDOW_SIZE)
browser.show()

In [ ]:
# Optionally load sections saved in a previous session and append to current browser
# Run this cell if you want to resume where you left off
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        previous = [json.loads(line) for line in f if line.strip()]
    browser.saved = previous
    browser._refresh_counter()
    nc = sum(1 for s in previous if s["type"] == "chosen")
    nr = sum(1 for s in previous if s["type"] == "rejected")
    print(f"Restored {len(previous)} sections from {OUTPUT_FILE} ({nc} chosen, {nr} rejected)")
else:
    print(f"No existing file at {OUTPUT_FILE} — starting fresh")